In [40]:
# ====================== 配置区 ======================
from pathlib import Path
import pandas as pd, numpy as np, json, ast

ROOT = Path("./results")
SETTING = "openset"   # "gcd" | "openset"

# 要纳入平均的“数据集白名单”（按需改；不存在的自动跳过）
DATASETS_WHITELIST = [
    "20NG","DBPedia","TREC","Yahoo",
    "banking","clinc","ecdt","ele","hwu","mcid",
    "news","stackoverflow","thucnews",
]

# 指标：与之前一致
METRICS_MAP = {
    "gcd":     ["K-ACC", "N-ACC"],
    "openset": ["K-F1", "N-F1"],
}

# 方法排序（没列出的按字母序排后面）
# METHOD_ORDER_MAP = {
#     "gcd": ["DeepAligned", "GeoID", "SDC", "DPN", "TAN", "LOOP", "ALUP"],
#     "openset": ["DOC", "DeepUNK", "ADB", "SCL", "AB", "KNNCon", "DyEn", 
#                 "MaxSoftmax", "OpenMax", "TemperatureScaling", "Mahalanobis", 
#                 "EnergyBased", "LogitNorm", "Entropy", "KLMatching", "MaxLogit"
#             ], 
# }
METHOD_ORDER_MAP = {
    "gcd": ["DeepAligned", "GeoID", "SDC", "DPN", "TAN", "LOOP", "ALUP"],
    "openset": ["DOC", "DeepUNK", "ADB", "SCL", "AB", "KNNCon", "DyEn", 
                 "OpenMax",  
                "EnergyBased",  "MaxLogit"
            ], 
}

# 名称归一化/重命名（确保展示名与 METHOD_ORDER_MAP 完全一致）
METHOD_RENAME = {
    # gcd
    "deepaligned":"DeepAligned","geoid":"GeoID","sdc":"SDC","dpn":"DPN","tan":"TAN","loop":"LOOP","alup":"ALUP",
    # openset（常规法）
    "doc":"DOC","deepunk":"DeepUNK","adb":"ADB","scl":"SCL","ab":"AB",
    "knncon":"KNNCon","knn-con":"KNNCon","knn_con":"KNNCon","dyen":"DyEn",
    # LLM-OOD/PLM_OOD 归一（这里只用于 umbrella 显示名；真正会被 detector 替换）
    "plm_ood":"LLM_OOD",
}

# KCR、LCR 候选与顺序（KCR×LCR=3×3=9）
KCR_ORDER = [0.25, 0.50, 0.75]
LCR_ORDER = [0.1, 0.5, 1.0]

# 输出
ROUND_DECIMALS = 2
HIGHLIGHT_TOP2 = True
OUTPUT_TEX = f"{SETTING}_avg_over_datasets_KCRxLCR.tex"

# 表格布局：默认 行=KCR→Method，列=按LCR分组（组内是各 metric）
LAYOUT = "KCR_rows_LCR_cols"   # 或 "both_in_cols"
# ==================== 配置区结束 ====================

In [41]:
# ---------- 辅助函数 ----------
def _normalize_key(s: str) -> str:
    return str(s).strip().lower().replace("_","").replace("-","")

def rename_method(raw_name: str, fallback_task: str) -> str:
    key = _normalize_key(raw_name if pd.notnull(raw_name) else fallback_task)
    return METHOD_RENAME.get(key, str(raw_name if pd.notnull(raw_name) else fallback_task))

def parse_args_field(s: str) -> dict:
    if pd.isna(s): return {}
    txt = str(s)
    try:
        return json.loads(txt)
    except Exception:
        pass
    try:
        fixed = txt.replace('""','"')
        if len(fixed)>=2 and fixed[0]=='"' and fixed[-1]=='"':
            fixed = fixed[1:-1]
        return json.loads(fixed)
    except Exception:
        pass
    try:
        return ast.literal_eval(txt)
    except Exception:
        return {}

def latex_escape_text(x) -> str:
    s = str(x)
    rep = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#",
           "_": r"\_", "{": r"\{", "}": r"\}", "~": r"\textasciitilde{}", "^": r"\textasciicircum{}"}
    return "".join(rep.get(ch, ch) for ch in s)

In [42]:
# ---------- 读取所有任务（含 plm_ood→Detector 拆分） ----------
base = ROOT / SETTING
metrics = METRICS_MAP[SETTING]
ALIASES_PLM = {"plmood","llmood"}

def extract_fold_type(args_str):
    obj = parse_args_field(args_str)
    return obj.get("fold_type")


# 任务列表（扫描）
task_list = sorted(p.name for p in base.iterdir() if p.is_dir() and (p/"results.csv").exists())

frames, missing = [], []
for task in task_list:
    csv_path = base / task / "results.csv"
    try:
        tdf = pd.read_csv(csv_path)
    except Exception:
        missing.append(str(csv_path)); continue

    tdf["__fold_type__"] = tdf["args"].apply(extract_fold_type)
    tdf = tdf[tdf["__fold_type__"].astype(str).str.lower() == "fold"].copy()
    
    # 统一 Method：一般方法 → 重命名；plm_ood → 用 args['Detector'/'Detecor'] 代替方法名
    if "method" in tdf.columns:
        def _method_from_row(row):
            raw_m = row.get("method", None)
            renamed = rename_method(raw_m, task)
            if any(_normalize_key(x) in ALIASES_PLM for x in (task, raw_m, renamed)):
                args_obj = parse_args_field(row.get("args",""))
                det = args_obj.get("Detector") or args_obj.get("Detecor")
                return det if det else "LLM-OOD"
            return renamed
        tdf["Method"] = tdf.apply(_method_from_row, axis=1)
    else:
        # 没有 method 列时，仅用目录名判断
        if _normalize_key(task) in ALIASES_PLM:
            def _det_from_args(row):
                args_obj = parse_args_field(row.get("args",""))
                det = args_obj.get("Detector") or args_obj.get("Detecor")
                return det if det else "LLM-OOD"
            tdf["Method"] = tdf.apply(_det_from_args, axis=1)
        else:
            tdf["Method"] = rename_method(task, task)

    # 基本字段
    tdf["dataset"] = tdf["dataset"].astype(str)
    tdf["KCR"] = tdf.get("known_cls_ratio", np.nan)
    tdf["LCR"] = tdf.get("labeled_ratio", np.nan)
    frames.append(tdf)

if not frames:
    raise FileNotFoundError("未找到任何 results.csv，或读取失败。\n" + "\n".join(missing))

df = pd.concat(frames, ignore_index=True)

# —— 仅保留 METHOD_ORDER_MAP 中列出的模型（白名单）
ALLOWED = [m for m in METHOD_ORDER_MAP.get(SETTING, []) if m]  # 去掉空值
if ALLOWED:
    df["Method"] = df["Method"].astype(str)
    df = df[df["Method"].isin(set(ALLOWED))].copy()
    if df.empty:
        raise ValueError(
            f"{SETTING}: 方法白名单过滤后无数据。请检查 METHOD_ORDER_MAP['{SETTING}'] "
            f"是否与实际 Method 名（含各 Detector 名）一致。"
        )

# 只留白名单数据集
wl = [d.lower() for d in DATASETS_WHITELIST]
df = df[df["dataset"].str.lower().isin(wl)].copy()
if df.empty:
    raise ValueError("白名单筛选后无数据。请检查 DATASETS_WHITELIST。")

# openset 的指标若是分数(<1)，转百分数（只对要导出的指标列）
if SETTING == "openset":
    for m in metrics:
        if m in df.columns:
            df[m] = df[m].apply(lambda v: (v*100.0) if (pd.notnull(v) and isinstance(v,(int,float)) and v<1.0) else v)

# ---------- 聚合：先跨 seed/fold，后跨数据集 ----------
required_cols = ["dataset","Method","KCR","LCR"] + metrics
miss = [c for c in required_cols if c not in df.columns]
if miss:
    raise KeyError(f"缺失列：{miss}")

# 1) 对同一 (dataset, Method, KCR, LCR) 先均值
g1 = (df[required_cols]
      .groupby(["dataset","Method","KCR","LCR"], as_index=False)
      .mean(numeric_only=True))

# 2) 再跨数据集均值：得到“平均结果表”
g2 = (g1.groupby(["Method","KCR","LCR"], as_index=False)
         .mean(numeric_only=True))   # 这里 metrics 列会被均值化

# ---------- 排序（Method 按配置；KCR/LCR 按顺序） ----------
# KCR/LCR 分类顺序
g2["KCR"] = pd.Categorical(g2["KCR"], categories=KCR_ORDER, ordered=True)
g2["LCR"] = pd.Categorical(g2["LCR"], categories=LCR_ORDER, ordered=True)

# Method 排序
present_methods = set(g2["Method"].astype(str).unique().tolist())
whitelist = METHOD_ORDER_MAP.get(SETTING, []) or []
method_order = [m for m in whitelist if m in present_methods]
g2["Method"] = pd.Categorical(g2["Method"], categories=method_order, ordered=True)
# 保险：把不在白名单里的行（类别为 NaN）去掉
g2 = g2.dropna(subset=["Method"])

g2 = g2.sort_values(["KCR","Method","LCR"]).reset_index(drop=True)

# ---------- 透视成最终表（列：KCR → LCR → Metric；行：Method） ----------
metrics_to_use = metrics[:]  # 保序
long = g2.melt(id_vars=["Method","KCR","LCR"],
               value_vars=metrics_to_use,
               var_name="Metric", value_name="value")

# 行=Method；列=(KCR, LCR, Metric)
pivot = long.pivot_table(index=["Method"],
                         columns=["KCR","LCR","Metric"],
                         values="value",
                         aggfunc="mean")

# 只保留真实存在的三元列，并按 (KCR_ORDER, LCR_ORDER, metrics_to_use) 排序
existing_cols = set(map(tuple, pivot.columns.tolist()))
ordered_cols = [(k, l, m)
                for k in KCR_ORDER
                for l in LCR_ORDER
                for m in metrics_to_use
                if (k, l, m) in existing_cols]
pivot = pivot.reindex(columns=pd.MultiIndex.from_tuples(ordered_cols))
pivot = pivot.round(ROUND_DECIMALS)

# ---------- 高亮：每一列(某KCR,LCR,Metric)内对各 Method 排名 ----------
def highlight_col(col_ser: pd.Series) -> pd.Series:
    out = col_ser.astype(object)
    ser = pd.to_numeric(col_ser, errors="coerce")
    order = ser.sort_values(ascending=False, kind="mergesort")
    if len(order) >= 1 and pd.notnull(order.iloc[0]):
        out.loc[order.index[0]] = "\\textbf{" + f"{order.iloc[0]:.{ROUND_DECIMALS}f}" + "}"
    if len(order) >= 2 and pd.notnull(order.iloc[1]):
        out.loc[order.index[1]] = "\\underline{" + f"{order.iloc[1]:.{ROUND_DECIMALS}f}" + "}"
    # 其余
    for idxi, v in ser.items():
        if pd.isnull(v):
            out.loc[idxi] = ""
        elif out.loc[idxi] == col_ser.loc[idxi]:
            out.loc[idxi] = f"{v:.{ROUND_DECIMALS}f}"
    return out

formatted = pd.DataFrame(index=pivot.index)
for trip in ordered_cols:
    formatted[trip] = highlight_col(pivot[trip])

# ---------- 组装 LaTeX (table* + resize)：三层表头 KCR → LCR → Metric ----------
# 1) 列规格：左侧 Method 一列；每个 KCR 组占若干列，中间用竖线分隔
def count_cols_for_kcr(k):
    return sum(1 for (kk, ll, mm) in ordered_cols if kk == k)

group_specs = ["".join("r" for _ in range(count_cols_for_kcr(k))) for k in KCR_ORDER if count_cols_for_kcr(k) > 0]
colspec = "l|" + "|".join(group_specs) if group_specs else "l"

# 2) 表头第一行：KCR 组
head_line1_parts = ["Method"]
for i, k in enumerate([k for k in KCR_ORDER if count_cols_for_kcr(k) > 0]):
    ncols = count_cols_for_kcr(k)
    align = "c|" if i < len([kk for kk in KCR_ORDER if count_cols_for_kcr(kk) > 0]) - 1 else "c"
    head_line1_parts.append(f"\\multicolumn{{{ncols}}}{{{align}}}{{KCR={k:.2f}}}")
head_line1 = " & ".join(head_line1_parts) + " \\\\"

# 3) 表头第二行：在每个 KCR 下，按 LCR 再分组
def count_cols_for_kcr_lcr(k, l):
    return sum(1 for (kk, ll, mm) in ordered_cols if kk == k and ll == l)

head_line2_parts = [""]
for k in KCR_ORDER:
    if count_cols_for_kcr(k) == 0:
        continue
    for j, l in enumerate([l for l in LCR_ORDER if count_cols_for_kcr_lcr(k, l) > 0]):
        n = count_cols_for_kcr_lcr(k, l)
        head_line2_parts.append(f"\\multicolumn{{{n}}}{{c}}{{LCR={l:.1f}}}")
head_line2 = " & ".join(head_line2_parts) + " \\\\"

# 4) 表头第三行：具体 metrics
head_line3 = " & ".join(
    [""] + [m for (k, l, m) in ordered_cols]
) + " \\\\ \\midrule"

# 5) 每行
def row_to_latex(method, row):
    method_str = latex_escape_text(method)
    cells = [method_str]
    for trip in ordered_cols:
        v = row.get(trip, "")
        cells.append(v if isinstance(v, str) else ("" if pd.isnull(v) else f"{v:.{ROUND_DECIMALS}f}"))
    return " & ".join(cells) + " \\\\"

body_lines = [row_to_latex(idx, formatted.loc[idx]) for idx in formatted.index]

# 6) 输出 LaTeX
env = "table*"; size_cmd = "\\small"; tabcol = 4
latex_table = "\n".join([
    f"\\begin{{{env}}}[t]",
    "\\centering",
    size_cmd,
    f"\\setlength{{\\tabcolsep}}{{{tabcol}pt}}",
    "\\resizebox{\\textwidth}{!}{%",
    f"\\begin{{tabular}}{{{colspec}}}",
    "\\toprule",
    head_line1,
    head_line2,
    head_line3,
    "\n".join(body_lines),
    "\\bottomrule",
    "\\end{tabular}}",
    f"\\end{{{env}}}"
])

Path(OUTPUT_TEX).write_text(latex_table, encoding="utf-8")
print(f"[OK] 宽表生成：{Path(OUTPUT_TEX).resolve()}")


[OK] 宽表生成：/ssd/code/bolt/openset_avg_over_datasets_KCRxLCR.tex


/tmp/ipykernel_4065123/4118545314.py:118: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = long.pivot_table(index=["Method"],
